In [1]:
import pandas as pd
import os

os.chdir('/Users/ruchapatwardhan/Desktop/IDX Internship')

In [2]:
files = sorted([f for f in os.listdir('raw') if f.startswith('CRMLSListing') and f.endswith('.csv')])

print(f"Files found: {len(files)}")

listings = pd.concat([pd.read_csv(f'raw/{f}') for f in files], ignore_index=True)

print(f"Total rows after concatenation: {listings.shape[0]}")
print(f"Total columns: {listings.shape[1]}")

Files found: 25
Total rows after concatenation: 782537
Total columns: 84


In [3]:
print(f"Rows before Residential filter: {listings.shape[0]}")

listings = listings[listings['PropertyType'] == 'Residential']

print(f"Rows after Residential filter: {listings.shape[0]}")

Rows before Residential filter: 782537
Rows after Residential filter: 494622


In [4]:
listings.to_csv('listings_combined.csv', index=False)
print("Saved listings_combined.csv successfully!")

Saved listings_combined.csv successfully!


In [5]:
files = sorted([f for f in os.listdir('raw') if f.startswith('CRMLSListing') and f.endswith('.csv')])
print(f"Files found: {len(files)}")

listings = pd.concat([pd.read_csv(f'raw/{f}', low_memory=False) for f in files], ignore_index=True)
print(f"Total rows after concatenation: {listings.shape[0]}")
print(f"Total columns: {listings.shape[1]}")

Files found: 25
Total rows after concatenation: 782537
Total columns: 84


In [6]:
print(f"Rows before Residential filter: {listings.shape[0]}")
listings = listings[listings['PropertyType'] == 'Residential']
print(f"Rows after Residential filter: {listings.shape[0]}")

Rows before Residential filter: 782537
Rows after Residential filter: 494622


In [7]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url)
mortgage.columns = ['date', 'rate_30yr_fixed']
mortgage['date'] = pd.to_datetime(mortgage['date'])
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index()

listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')
listings = listings.merge(mortgage_monthly, on='year_month', how='left')
print(listings['rate_30yr_fixed'].isnull().sum())

0


In [8]:
listings.to_csv('listings_combined.csv', index=False)
print("Saved listings_combined.csv with mortgage rates!")

Saved listings_combined.csv with mortgage rates!


In [9]:
os.chdir('/Users/ruchapatwardhan/Desktop/IDX Internship')
listings = pd.read_csv('listings_combined.csv', low_memory=False)
print(listings.shape)

(494622, 86)


In [10]:
cols_before = listings.shape[1]
listings = listings.dropna(axis=1, how='all')
cols_after = listings.shape[1]
print(f"Columns before: {cols_before}")
print(f"Columns after: {cols_after}")
print(f"Columns dropped: {cols_before - cols_after}")

Columns before: 86
Columns after: 78
Columns dropped: 8


In [11]:
date_cols = ['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'ContractStatusChangeDate']
for col in date_cols:
    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')
        print(f"{col} converted successfully")

ListingContractDate converted successfully
PurchaseContractDate converted successfully
CloseDate converted successfully
ContractStatusChangeDate converted successfully


In [12]:
listings['invalid_list_price'] = listings['ListPrice'] <= 0
listings['invalid_living_area'] = listings['LivingArea'] <= 0
listings['missing_coords'] = listings['Latitude'].isnull() | listings['Longitude'].isnull()
listings['wrong_longitude'] = listings['Longitude'] > 0

print(f"Invalid ListPrice: {listings['invalid_list_price'].sum()}")
print(f"Invalid LivingArea: {listings['invalid_living_area'].sum()}")
print(f"Missing coordinates: {listings['missing_coords'].sum()}")
print(f"Wrong longitude: {listings['wrong_longitude'].sum()}")

Invalid ListPrice: 0
Invalid LivingArea: 327
Missing coordinates: 79632
Wrong longitude: 61


In [13]:
listings.to_csv('listings_cleaned.csv', index=False)
print(f"Saved listings_cleaned.csv")
print(f"Final shape: {listings.shape}")

Saved listings_cleaned.csv
Final shape: (494622, 82)


In [14]:
listings['price_per_sqft'] = listings['ListPrice'] / listings['LivingArea']
print("Price metrics created!")

Price metrics created!


In [15]:
listings['ListingContractDate'] = pd.to_datetime(listings['ListingContractDate'])
listings['PurchaseContractDate'] = pd.to_datetime(listings['PurchaseContractDate'])
listings['CloseDate'] = pd.to_datetime(listings['CloseDate'])

listings['year'] = listings['ListingContractDate'].dt.year
listings['month'] = listings['ListingContractDate'].dt.month
listings['yr_mo'] = listings['ListingContractDate'].dt.to_period('M')

listings['listing_to_contract_days'] = (listings['PurchaseContractDate'] - listings['ListingContractDate']).dt.days
listings['contract_to_close_days'] = (listings['CloseDate'] - listings['PurchaseContractDate']).dt.days

print("Time metrics created!")

Time metrics created!


In [16]:
segment = listings.groupby('CountyOrParish').agg(
    median_list_price=('ListPrice', 'median'),
    avg_price_per_sqft=('price_per_sqft', 'median'),
    avg_days_on_market=('DaysOnMarket', 'median'),
    total_units=('ListPrice', 'count')
).reset_index().sort_values('median_list_price', ascending=False)

print(segment.head(10))

   CountyOrParish  median_list_price  avg_price_per_sqft  avg_days_on_market  \
45      San Mateo          1648000.0         1005.659892                11.0   
47    Santa Clara          1499000.0          923.290203                 9.0   
22          Marin          1297000.0          742.432016                13.0   
48     Santa Cruz          1249999.5          758.820704                13.0   
31         Orange          1200000.0          687.727579                10.0   
42  San Francisco          1095000.0          872.052402                12.0   
0         Alameda           998000.0          662.380989                12.0   
28       Monterey           993000.0          633.367662                12.0   
20    Los Angeles           950000.0          631.596948                12.0   
41      San Diego           910000.0          606.408946                 9.0   

    total_units  
45         9030  
47        23603  
22          207  
48         4160  
31        47311  
42         

In [17]:
listings.to_csv('listings_featured.csv', index=False)
print(f"Saved listings_featured.csv")
print(f"Final shape: {listings.shape}")

Saved listings_featured.csv
Final shape: (494622, 88)


In [18]:
def flag_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (df[col] < lower) | (df[col] > upper)

listings['outlier_list_price'] = flag_outliers(listings, 'ListPrice')
listings['outlier_living_area'] = flag_outliers(listings, 'LivingArea')
listings['outlier_days_on_market'] = flag_outliers(listings, 'DaysOnMarket')

print(f"ListPrice outliers: {listings['outlier_list_price'].sum()}")
print(f"LivingArea outliers: {listings['outlier_living_area'].sum()}")
print(f"DaysOnMarket outliers: {listings['outlier_days_on_market'].sum()}")

ListPrice outliers: 41438
LivingArea outliers: 24548
DaysOnMarket outliers: 46558


In [19]:
print(f"Rows before outlier filter: {listings.shape[0]}")

listings_filtered = listings[
    ~listings['outlier_list_price'] & 
    ~listings['outlier_living_area'] & 
    ~listings['outlier_days_on_market']
]

print(f"Rows after outlier filter: {listings_filtered.shape[0]}")
print(f"Rows removed: {listings.shape[0] - listings_filtered.shape[0]}")

Rows before outlier filter: 494622
Rows after outlier filter: 404759
Rows removed: 89863


In [21]:
listings.to_csv('listings_flagged.csv', index=False)
listings_filtered.to_csv('listings_final.csv', index=False)

print(f"Saved listings_flagged.csv: {listings.shape}")
print(f"Saved listings_final.csv: {listings_filtered.shape}")

Saved listings_flagged.csv: (494622, 91)
Saved listings_final.csv: (404759, 91)
